# Online News Popularity: Reproducible Macro-F1 Ensemble

This notebook trains a leakage-safe repeated-CV ensemble using only the organizer CSV files. It combines CatBoost and LightGBM probabilities, selects blend weights from out-of-fold (OOF) predictions, calibrates the multiclass decision rule for Macro F1, and validates the final submission schema. No test labels, external datasets, or pretrained models are used.

In [ ]:
from pathlib import Path
import itertools
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

from ml_core import CLASSES, CLASS_TO_INDEX, TARGET_COLUMN, build_features, fit_fold

ROOT = Path.cwd()
DATA = ROOT / 'dataset'
OUT = ROOT / 'submissions'
OUT.mkdir(exist_ok=True)
SEEDS = (42, 2026, 7, 1337)
N_SPLITS = 5
LABELS = np.asarray(CLASSES)
assert all((DATA / name).is_file() for name in ('train.csv', 'test.csv', 'sample_submission.csv'))

## Data and feature integrity

Identifiers and the target are excluded from model inputs. Every engineered feature is calculated independently per row, so validation and test rows cannot influence fitted preprocessing statistics.

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
assert list(sample.columns) == ['id', TARGET_COLUMN]
assert test['id'].equals(sample['id']) and test['id'].is_unique
assert train['id'].is_unique and not (set(train['id']) & set(test['id']))
assert train[TARGET_COLUMN].isin(CLASSES).all()
y = train[TARGET_COLUMN].map(CLASS_TO_INDEX).to_numpy(dtype=int)
X = build_features(train, kind='v2')
X_test = build_features(test, kind='v2')
assert list(X.columns) == list(X_test.columns)
assert 'id' not in X and TARGET_COLUMN not in X
print('train/test/features:', train.shape, test.shape, X.shape[1])
print('target counts:', train[TARGET_COLUMN].value_counts().sort_index().to_dict())

## Repeated stratified OOF training

The ensemble emphasizes complementary class-weighting regimes. Early stopping sees only the current held-out fold. Each training row receives predictions from a model that did not fit that row, while test probabilities are averaged across folds and seeds.

In [ ]:
VARIANTS = {
    'catboost_bal': ('catboost', 'balanced'),
    'catboost_sqrt': ('catboost', 'sqrt_balanced'),
    'lgbm_bal': ('lgbm', 'balanced'),
}

oof_by_model = {}
test_by_model = {}
fold_log = []
for name, (model_name, weight_mode) in VARIANTS.items():
    seed_oof, seed_test = [], []
    for seed in SEEDS:
        splitter = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed)
        oof = np.zeros((len(train), len(CLASSES)), dtype=np.float32)
        heldout = np.zeros((len(test), len(CLASSES)), dtype=np.float32)
        seen = np.zeros(len(train), dtype=bool)
        for fold, (fit_idx, valid_idx) in enumerate(splitter.split(X, y)):
            valid_p, test_p, best_iteration = fit_fold(
                model_name, X.iloc[fit_idx], y[fit_idx], X.iloc[valid_idx],
                y[valid_idx], X_test, seed=seed * 1000 + fold,
                weight_mode=weight_mode, use_gpu=False,
            )
            assert not seen[valid_idx].any()
            seen[valid_idx] = True
            oof[valid_idx] = valid_p
            heldout += np.asarray(test_p, dtype=np.float32) / N_SPLITS
            score = f1_score(y[valid_idx], np.argmax(valid_p, axis=1), average='macro')
            fold_log.append((name, seed, fold, score, best_iteration))
            print(name, seed, fold, f'Macro F1={score:.5f}', 'iteration', best_iteration)
        assert seen.all() and np.isfinite(oof).all() and np.isfinite(heldout).all()
        seed_oof.append(oof)
        seed_test.append(heldout)
    oof_by_model[name] = np.mean(seed_oof, axis=0)
    test_by_model[name] = np.mean(seed_test, axis=0)
    score = f1_score(y, oof_by_model[name].argmax(axis=1), average='macro')
    print(name, 'seed-averaged OOF Macro F1:', score)

## OOF-only blend search

A compact simplex grid chooses model weights using OOF Macro F1. The test probabilities are untouched during selection.

In [ ]:
names = list(VARIANTS)
grid = np.linspace(0.0, 1.0, 11)
candidates = []
for w0 in grid:
    for w1 in grid:
        w2 = 1.0 - w0 - w1
        if w2 < -1e-12:
            continue
        weights = np.array([w0, w1, max(0.0, w2)])
        p = sum(weights[i] * oof_by_model[name] for i, name in enumerate(names))
        candidates.append((f1_score(y, p.argmax(axis=1), average='macro'), weights))
raw_score, blend_weights = max(candidates, key=lambda item: item[0])
blend_oof = sum(blend_weights[i] * oof_by_model[name] for i, name in enumerate(names))
blend_test = sum(blend_weights[i] * test_by_model[name] for i, name in enumerate(names))
print('weights:', dict(zip(names, blend_weights.round(2))))
print('raw blend OOF Macro F1:', raw_score)

## Macro-F1 decision calibration

Multiclass argmax is not generally optimal for Macro F1. Coordinate ascent selects bounded class multipliers on OOF predictions. A nested five-way check must show improvement on a majority of held-out partitions before calibration is accepted.

In [ ]:
def macro_f1(y_true, probabilities, multipliers):
    return f1_score(y_true, (probabilities * multipliers).argmax(axis=1),
                    labels=range(len(CLASSES)), average='macro', zero_division=0)

def tune_multipliers(probabilities, targets):
    multipliers = np.ones(len(CLASSES), dtype=float)
    search = np.linspace(0.70, 1.30, 25)
    for _ in range(4):
        changed = False
        for cls in range(len(CLASSES)):
            trials = []
            for value in search:
                candidate = multipliers.copy()
                candidate[cls] = value
                trials.append((macro_f1(targets, probabilities, candidate), value))
            _, value = max(trials)
            changed |= value != multipliers[cls]
            multipliers[cls] = value
        if not changed:
            break
    return multipliers

gate = StratifiedKFold(5, shuffle=True, random_state=90210)
improvements = []
for fit_idx, valid_idx in gate.split(blend_oof, y):
    m = tune_multipliers(blend_oof[fit_idx], y[fit_idx])
    before = macro_f1(y[valid_idx], blend_oof[valid_idx], np.ones(5))
    after = macro_f1(y[valid_idx], blend_oof[valid_idx], m)
    improvements.append(after - before)
multipliers = tune_multipliers(blend_oof, y) if sum(x > 0 for x in improvements) >= 3 else np.ones(5)
final_oof_score = macro_f1(y, blend_oof, multipliers)
print('nested improvements:', np.round(improvements, 5))
print('multipliers:', dict(zip(CLASSES, multipliers.round(3))))
print('calibrated OOF Macro F1:', final_oof_score)
print('per-class F1:', dict(zip(CLASSES, np.round(f1_score(y, (blend_oof * multipliers).argmax(1), average=None), 4))))

## Optional balanced-test assignment

The competition test set may be deliberately balanced across its five classes. If that fact is explicitly stated by the organizer, set `TEST_IS_KNOWN_BALANCED=True`. The Hungarian assignment then finds the maximum-likelihood labeling with exactly 500 predictions per class. Do not enable this merely by looking at leaderboard scores.

In [ ]:
TEST_IS_KNOWN_BALANCED = False

def quota_assignment(probabilities, counts):
    from scipy.optimize import linear_sum_assignment
    slots = np.repeat(np.arange(len(CLASSES)), counts)
    if len(slots) != len(probabilities):
        raise ValueError('Quota counts must sum to the number of rows')
    costs = -np.log(np.clip(probabilities[:, slots], 1e-12, 1.0))
    rows, columns = linear_sum_assignment(costs)
    prediction = np.empty(len(probabilities), dtype=int)
    prediction[rows] = slots[columns]
    return prediction

final_test_probabilities = blend_test * multipliers
if TEST_IS_KNOWN_BALANCED:
    assert len(test) % len(CLASSES) == 0
    per_class = len(test) // len(CLASSES)
    prediction = quota_assignment(final_test_probabilities, [per_class] * len(CLASSES))
else:
    prediction = final_test_probabilities.argmax(axis=1)
print('prediction counts:', dict(zip(CLASSES, np.bincount(prediction, minlength=5))))

## Write and independently validate the submission

In [ ]:
submission = pd.DataFrame({'id': test['id'].to_numpy(), TARGET_COLUMN: LABELS[prediction]})
assert list(submission.columns) == ['id', TARGET_COLUMN]
assert len(submission) == len(test)
assert submission['id'].equals(test['id']) and submission['id'].equals(sample['id'])
assert submission['id'].is_unique and submission[TARGET_COLUMN].isin(CLASSES).all()
assert not submission.isna().any().any()
output_path = OUT / 'submission.csv'
submission.to_csv(output_path, index=False)
report = {
    'oof_macro_f1': float(final_oof_score),
    'blend_weights': dict(zip(names, blend_weights.tolist())),
    'class_multipliers': dict(zip(CLASSES, multipliers.tolist())),
    'balanced_test_constraint': TEST_IS_KNOWN_BALANCED,
    'prediction_counts': submission[TARGET_COLUMN].value_counts().sort_index().to_dict(),
}
(OUT / 'report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('validated submission:', output_path)
print(json.dumps(report, indent=2))

## Interpretation

The reported OOF score is the reproducible local estimate. A Kaggle score is measured on different hidden labels and can be higher or lower. Keep the generated `report.json`, exact CSV hash, package versions, and Kaggle submission record together when reporting a leaderboard result.